# 01 — Tokenizer

Train a **Byte-Level BPE (Byte Pair Encoding)** tokenizer for A.N.T.H.O.R.
Saved directly to **Google Drive**.

> Runs on **Google Colab** (free T4 tier).

## 1. Install dependencies

In [ ]:
!pip install -q tokenizers datasets transformers

## 2. Mount Google Drive (MANDATORY)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

In [ ]:
# ─── Tokenizer config ───
VOCAB_SIZE   = 16_384
MIN_FREQUENCY = 2

# ─── Data config ───
DATASET_NAME   = "HuggingFaceFW/fineweb-edu"
DATASET_CONFIG = "sample-10BT"
DATASET_SPLIT  = "train"
SAMPLE_SIZE    = 10_000

# ─── Google Drive paths (MANDATORY) ───
DRIVE_BASE     = "/content/drive/MyDrive/ANTHOR"
TOKENIZER_PATH = f"{DRIVE_BASE}/tokenizer/tokenizer.json"

import os
os.makedirs(f"{DRIVE_BASE}/tokenizer", exist_ok=True)

## 4. Load dataset

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET_NAME} ({DATASET_CONFIG}/{DATASET_SPLIT})...")
ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT, streaming=True)

docs = []
for i, example in enumerate(ds):
    if i >= SAMPLE_SIZE:
        break
    docs.append(example["text"])
    if (i + 1) % 2000 == 0:
        print(f"  Loaded {i + 1}/{SAMPLE_SIZE}")

total_bytes = sum(len(d.encode("utf-8")) for d in docs)
print(f"\nLoaded {len(docs)} docs, ~{total_bytes / 1e6:.1f} MB")

## 5. Train & Save Tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
import time

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()])
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

print(f"Training BPE (vocab_size={VOCAB_SIZE})...")
t0 = time.time()
tokenizer.train_from_iterator(docs, trainer)
print(f"Done in {time.time() - t0:.1f}s, vocab_size={tokenizer.get_vocab_size()}")

# ── Save to Google Drive (MANDATORY) ──
os.makedirs(os.path.dirname(TOKENIZER_PATH), exist_ok=True)
tokenizer.save(TOKENIZER_PATH)
print(f"Saved to Drive: {TOKENIZER_PATH}")

## 6. Verify saved tokenizer

In [ ]:
import os
loaded = Tokenizer.from_file(TOKENIZER_PATH)
print(f"Loaded tokenizer from Drive: vocab_size={loaded.get_vocab_size()}")
test = loaded.encode("Xin chào Việt Nam!")
print(f"Test encode: {test.ids}")
print(f"Test decode: {loaded.decode(test.ids)}")

---

✅ **Next:** `02_data_pipeline.ipynb` (saves to Google Drive)

> **Note:** All data/tokenizer/models are now **Google Drive only** — no local storage.